# TalentDesk, Module 2 Section 2 Lab (Exercise): MCP Config, Structured Errors, and Retry

A hands-on exercise on wiring **MCP servers** into TalentDesk and making their tools **fail well**.
It combines the two M2S2 skills: declaring MCP servers with `${VAR}` credentials and connecting
several at once (Lab 1), and returning **structured errors** so a retry harness retries transient
failures and escalates the rest (Lab 2). You fill in four short `TODO` blocks; everything else is
provided. The expander, validator, error taxonomy, and retry harness are all testable offline, and a
full solution is at the end. Runs **Sonnet** (`claude-sonnet-4-6`).

## The real-world scenario

TalentDesk's data lives in several systems: an **ATS database** (candidates and stages), a
**background-check service**, and a scheduling API. **MCP** lets each become a set of tools the agent
can call, and `.mcp.json` is how you declare them so the whole team shares one config. The trick is
doing it **securely**: the config is committed to Git, but the secrets are not; they come from
environment variables at connect time.

And these tools fail. The ATS is briefly overloaded, a recruiter passes a malformed candidate id, an
action exceeds hiring policy, the agent lacks permission to the background service. If every failure
looks the same, the agent either retries things it never should or gives up on things it could have
recovered. A **structured error** says what kind of failure it was and whether retrying could help.

The question this lab answers: **how do you declare multiple MCP servers and keep their credentials
out of source control, and how do you shape tool errors so the agent retries the transient ones and
escalates the rest?**

## Objectives

- Configure MCP servers in **`.mcp.json`** and reference credentials with **`${VAR}`** so secrets
  stay in the environment, not in Git.
- Validate a config to catch a **hardcoded secret**, and connect **multiple servers** so all their
  tools are available at once.
- Signal failure with **`isError`**, categorise it (transient, validation, business, permission) with
  an **`isRetryable`** flag, and keep it distinct from a valid empty result.
- Build **retry logic** that retries only transient errors and routes the rest to the right action.

## The outcome you should reach

By the end you will have:

- an env-var expander that resolves `${VAR}` and `${VAR:-default}`, and a validator that flags a
  hardcoded secret and passes a `${VAR}` config;
- a tool that returns the four structured error categories with the right `isRetryable` values;
- and a retry harness that recovers a transient failure after a backoff and refuses to retry the
  others (or a valid empty result).

Target time: **20 to 30 minutes.** Four small `TODO` blocks, all testable offline. The live
multi-server run needs a real key and Node.js 18+.

## How to run

Run top to bottom. The expander, validator, error taxonomy, and retry harness are pure Python and run
anywhere. The `.mcp.json` block is config reference for a real repo. The multi-server run calls
Claude, so paste a real key into **Setup 2/3** and re-run from the top; **Node.js 18+** must be
installed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages for the runnable Agent SDK section. Claude Code itself is
installed separately in your terminal for the config section. The Agent SDK also needs Node.js 18+,
which cannot be pip-installed; the offline cells do not need it.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` so the async
multi-server run can be called like a normal function.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # read env vars (this lab is all about env vars)
import re                                       # expand ${VAR} and detect secrets
import sys                                      # detect Windows (it needs a special event loop)
import json                                     # build and read configs and structured errors
import time                                     # a tiny sleep for the backoff demo
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv             #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the agent will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}
    def worker():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:    box["value"] = loop.run_until_complete(make_coro())
        except Exception as e: box["error"] = e
        finally: loop.close()
    t = threading.Thread(target=worker); t.start(); t.join()
    if "error" in box: raise box["error"]
    return box.get("value")

print("live model calls:", "ON" if RUN_LIVE else "OFF (config + retry logic run offline)")

### MCP servers, and where they are configured

An **MCP server** exposes capabilities through three primitives: **tools** (actions the agent can
call), **resources** (read-only data loaded on demand), and **prompts** (reusable templates). Claude
Code reads server config from **`.mcp.json`** at the project root (shared and committed to Git) and
**`~/.claude.json`** (your personal, user-level servers). Secrets never go in either file directly;
you reference them with `${VAR}`, expanded from your shell at connect time.

**Config reference (for a real repo, not run by the notebook).** One `.mcp.json` can declare
several servers, and all their tools become available at once. The `${VAR}` and `${VAR:-default}`
references keep the file safe to commit:

```json
{
  "mcpServers": {
    "ats-db": {
      "type": "stdio",
      "command": "npx",
      "args": ["-y", "@talentdesk/ats-mcp", "--dsn", "${ATS_DSN}"]
    },
    "background-svc": {
      "type": "http",
      "url": "${BGCHECK_BASE_URL:-https://api.bgcheck.example.com}/mcp",
      "headers": { "Authorization": "Bearer ${BGCHECK_API_KEY}" }
    }
  }
}
```

From the CLI you can add the same servers without editing JSON by hand, and check status with `/mcp`:

```bash
claude mcp add --transport stdio --scope project ats-db \
  -- npx -y @talentdesk/ats-mcp --dsn "${ATS_DSN}"
claude mcp add --transport http --scope project background-svc \
  "https://api.bgcheck.example.com/mcp" --header "Authorization: Bearer ${BGCHECK_API_KEY}"
```

---

### 🎯 Part A - declare servers, keep secrets safe

**This cell:** the same `.mcp.json` as a Python dict (provided). Generating config in code is
handy for scripts and tests; the shape is exactly what Claude Code reads.

In [ ]:
# ===== the .mcp.json as a Python dict (provided) =====
MCP_CONFIG = {
    "mcpServers": {
        "ats-db": {"type": "stdio", "command": "npx",
                   "args": ["-y", "@talentdesk/ats-mcp", "--dsn", "${ATS_DSN}"]},
        "background-svc": {"type": "http",
                           "url": "${BGCHECK_BASE_URL:-https://api.bgcheck.example.com}/mcp",
                           "headers": {"Authorization": "Bearer ${BGCHECK_API_KEY}"}},
    }
}
print(json.dumps(MCP_CONFIG, indent=2))

**TODO 1 (about 5 minutes).** Complete the `repl()` inside `expand()`, the env-var resolver. For
one `${VAR}` or `${VAR:-default}` reference: return the environment value if set; else the default if
one was given; else leave the literal untouched (a case Claude Code would warn about). The regex and
`expand_config()` are provided.

In [ ]:
# ===== TODO 1 - expand ${VAR} and ${VAR:-default} like Claude Code does =====
def expand(value):                                 # expand env references inside one string
    def repl(m):                                   #   handle one ${...} match
        var, default = m.group(1), m.group(3)      #     the name and the optional default
        val = os.environ.get(var)                  #     look it up in the environment
        # 👉 TODO 1a: if val is not None, return val
        # 👉 TODO 1b: if default is not None, return default
        # 👉 TODO 1c: otherwise return m.group(0) (leave the literal)
        return m.group(0)                          #     replace this line per the steps above
    return re.sub(r"\$\{([A-Za-z_][A-Za-z0-9_]*)(:-([^}]*))?\}", repl, value)

def expand_config(obj):                            # walk the config and expand every string (provided)
    if isinstance(obj, dict):  return {k: expand_config(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [expand_config(v) for v in obj]
    if isinstance(obj, str):   return expand(obj)
    return obj

os.environ["BGCHECK_API_KEY"] = "bgk_demo_123"     # pretend this came from your shell / secret store
print(json.dumps(expand_config(MCP_CONFIG), indent=2))   # key filled, url uses default, dsn stays literal

**Self-check (offline).** Verifies the three resolution cases.

In [ ]:
# ===== self-check for TODO 1 =====
os.environ["BGCHECK_API_KEY"] = "bgk_demo_123"
assert expand("Bearer ${BGCHECK_API_KEY}") == "Bearer bgk_demo_123", "set var should resolve"
assert expand("${BGCHECK_BASE_URL:-https://d.example.com}/mcp") == "https://d.example.com/mcp", "use the default"
assert expand("--dsn ${ATS_DSN}") == "--dsn ${ATS_DSN}", "unset, no default -> literal"
print("TODO 1 checks passed")

**TODO 2 (about 5 minutes).** Complete `validate()`. For each server, add an issue if it has
neither a `command` nor a `url`, and add an issue if its JSON contains something that looks like a
**hardcoded secret** (use the provided `SECRET_LOOKS_LIKE` pattern). Return the issues, or an "ok"
message if there are none.

In [ ]:
# ===== TODO 2 - validate a config: structure and no hardcoded secrets =====
SECRET_LOOKS_LIKE = re.compile(r"(ghp_|sk-ant-|bgk_|AKIA)[A-Za-z0-9_-]{6,}")   # rough secret shapes

def validate(cfg):                                 # config dict -> list of problems
    issues = []
    if "mcpServers" not in cfg:
        return ["missing 'mcpServers'"]
    for name, s in cfg["mcpServers"].items():
        # 👉 TODO 2a: if neither "command" nor "url" is in s, append f"{name}: needs a 'command' or 'url'"
        # 👉 TODO 2b: if SECRET_LOOKS_LIKE.search(json.dumps(s)), append f"{name}: HARDCODED SECRET - use ${{VAR}}"
        pass                                       #   replace with the two checks
    return issues or ["ok: structure valid, no hardcoded secrets"]

print("good config:", validate(MCP_CONFIG))
bad = {"mcpServers": {"x": {"type": "http", "url": "https://a",
                            "headers": {"Authorization": "Bearer bgk_live_ABCDEF"}}}}
print("bad config: ", validate(bad))

**Self-check (offline).**

In [ ]:
# ===== self-check for TODO 2 =====
assert validate(MCP_CONFIG) == ["ok: structure valid, no hardcoded secrets"], "the ${VAR} config is clean"
bad = {"mcpServers": {"x": {"url": "https://a", "headers": {"Authorization": "Bearer bgk_live_ABCDEF"}}}}
assert any("HARDCODED SECRET" in i for i in validate(bad)), "a raw secret must be flagged"
assert any("needs a 'command'" in i for i in validate({"mcpServers": {"y": {"type": "http"}}}))
print("TODO 2 checks passed")

**This cell:** **MCP resources** (provided). Unlike tools (actions), a resource is read-only data
loaded on demand with an `@` reference, so the agent gets exactly the context it needs and nothing
more.

In [ ]:
# ===== MCP resources: load context on demand (provided) =====
print("tool     -> an action the agent calls, e.g. mcp__ats-db__get_candidate")
print("resource -> read-only context, referenced with @, e.g. @ats-db:schema://candidates")
print("prompt   -> a reusable template, e.g. /mcp__ats-db__screen_candidate")
print("Referencing @ats-db:record://C1 loads just that candidate's context, not the whole ATS.")

---

### 🎯 Part B - errors the agent can act on

MCP has two error paths. **Protocol errors** (JSON-RPC) mean the message was bad; the client handles
those and the model never sees them. **Tool execution errors** mean the tool ran but failed; you
signal them with **`isError: true`** and the content goes back to the model so it can recover. Every
tool failure falls into one of four categories, and a **valid empty result** (`isError: false`,
`resultCount: 0`) is a success, not a failure.

**This cell:** the result builders (provided). `ok()` is a success with a `resultCount` so an
empty result is unambiguous; `err()` is a structured failure carrying `isError`, the message, the
`errorCategory`, `isRetryable`, and a recovery `description`.

In [ ]:
# ===== the structured result shapes (provided) =====
def ok(text, result_count=1):                      # a SUCCESS result (MCP: isError false)
    return {"isError": False, "resultCount": result_count,
            "content": [{"type": "text", "text": text}]}

def err(category, message, retryable, description): # a structured FAILURE result (MCP: isError true)
    return {"isError": True,
            "content": [{"type": "text", "text": message}],
            "errorCategory": category,              # transient / validation / business / permission
            "isRetryable": retryable,               # guides the retry logic
            "description": description}             # recovery guidance for the agent
print("result builders ready: ok(), err()")

**TODO 3 (about 5 minutes).** Complete the four failure modes in `candidate_lookup`. For each,
call `err()` with the right **category**, **isRetryable** flag, and a recovery **description**. Only
**transient** is retryable. The success and empty cases are provided.

In [ ]:
# ===== TODO 3 - one tool, four failure modes (plus success and empty) =====
def candidate_lookup(candidate_id, failure_mode="ok"):
    if failure_mode == "transient":                # temporary: the ATS is briefly overloaded
        # 👉 TODO 3a: return err("transient", "ATS temporarily unavailable.", <retryable?>, "<recovery>")
        pass
    if failure_mode == "validation":               # bad input format
        # 👉 TODO 3b: return err("validation", f"candidate_id '{candidate_id}' is malformed; expected like 'C1'.", <retryable?>, "<recovery>")
        pass
    if failure_mode == "business":                 # policy limit
        # 👉 TODO 3c: return err("business", "Action exceeds the approval policy for this requisition.", <retryable?>, "<recovery>")
        pass
    if failure_mode == "permission":               # access denied
        # 👉 TODO 3d: return err("permission", "Access denied to the background-check service.", <retryable?>, "<recovery>")
        pass
    if failure_mode == "empty":                    # VALID empty result (success!)
        return ok("No matching candidate.", result_count=0)
    return ok(f"candidate for {candidate_id}: Ana")   # normal success

for mode in ["transient", "validation", "business", "permission"]:
    r = candidate_lookup("C1", mode)
    print(f"{mode:11} isError={r['isError']} retryable={r['isRetryable']}  {r['description']}")

**Self-check (offline).** Only transient should be retryable, and the empty result must read as a
success.

In [ ]:
# ===== self-check for TODO 3 =====
assert candidate_lookup("C1", "transient")["isRetryable"] is True, "transient must be retryable"
for m in ["validation", "business", "permission"]:
    r = candidate_lookup("C1", m)
    assert r["isError"] is True and r["isRetryable"] is False, f"{m} must be a non-retryable error"
empty = candidate_lookup("C9", "empty")
assert empty["isError"] is False and empty["resultCount"] == 0, "empty is a success, not a failure"
print("TODO 3 checks passed")

**TODO 4 (about 5 minutes).** Complete `call_with_retry`. On each attempt: if the result is not
an error, return it (a success, including a valid empty result). If it is an error that is **not**
`isRetryable`, return it immediately with its guidance. Otherwise it is transient, so wait the backoff
and try again, up to `max_attempts`.

In [ ]:
# ===== TODO 4 - retry only what is retryable =====
def call_with_retry(make_call, max_attempts=4):    # make_call: a zero-arg function returning a result
    delay = 0.02                                    # a tiny starting backoff (seconds)
    for attempt in range(1, max_attempts + 1):
        result = make_call()
        # 👉 TODO 4a: if not result["isError"], print a success line and return result
        # 👉 TODO 4b: if not result.get("isRetryable"), print a "not retryable -> stop" line and return result
        # 👉 TODO 4c: otherwise print a "retrying" line, time.sleep(delay), then delay *= 2
        pass                                       #   replace with the three steps
    print("  gave up after", max_attempts, "attempts")
    return result

**Self-check (offline).** A transient failure that clears on the third attempt should recover;
the non-retryable categories should stop after one attempt; an empty result should return without
retrying.

In [ ]:
# ===== self-check for TODO 4 =====
_state = {"n": 0}
def flaky():                                        # transient twice, then succeeds
    _state["n"] += 1
    return candidate_lookup("C1", "transient" if _state["n"] < 3 else "ok")

print("transient case:")
res = call_with_retry(flaky)
assert res["isError"] is False and _state["n"] == 3, "should recover on the 3rd attempt"

for m in ["validation", "business", "permission"]:
    calls = {"n": 0}
    def once(mm=m):
        calls["n"] += 1
        return candidate_lookup("bad", mm)
    print(f"{m} case:")
    r = call_with_retry(once)
    assert calls["n"] == 1, f"{m} must not be retried"

empty_calls = {"n": 0}
def empty_once():
    empty_calls["n"] += 1
    return candidate_lookup("C9", "empty")
print("empty case:")
call_with_retry(empty_once)
assert empty_calls["n"] == 1, "a valid empty result must not be retried"
print("TODO 4 checks passed")

**This cell:** a compact **decision table** (provided): each category maps to one action, built
from the metadata so one harness handles every tool consistently.

In [ ]:
# ===== category -> action, straight from the metadata (provided) =====
ACTION = {"transient": "retry with backoff", "validation": "fix the input and recall",
          "business": "escalate to a human", "permission": "obtain credentials"}
for cat, act in ACTION.items():
    print(f"  {cat:11} -> {act}")
print("  (empty result -> accept it; do NOT retry)")

**This cell:** the runnable **multi-server** setup (provided, live). Two in-process servers (ATS
and background) go into one options object, and the background tool reads its credential from an
**environment variable** rather than a literal. All their tools are available at once, and the ATS
tool returns a structured error the model can read and adjust to. Offline it prints the expected
outcome.

In [ ]:
# ===== two in-process MCP servers, wired together (provided, live) =====
try:
    from claude_agent_sdk import (query, ClaudeAgentOptions, tool, create_sdk_mcp_server,
                                  AssistantMessage, ResultMessage, TextBlock, ToolUseBlock)
    SDK_OK = True

    ATS = {"C1": "interview", "C2": "screening"}

    @tool("get_candidate", "Get a candidate's stage by id like 'C1'.", {"candidate_id": str})
    async def ats_get(args):
        oid = args["candidate_id"]
        if not (len(oid) == 2 and oid[0] == "C" and oid[1:].isdigit()):     # malformed -> structured error
            payload = err("validation", f"candidate_id '{oid}' is malformed; expected like 'C1'.", False,
                          "Correct the candidate_id and call again.")
            return {"content": [{"type": "text", "text": json.dumps(payload)}], "isError": True}
        return {"content": [{"type": "text", "text": ATS.get(oid, "unknown candidate")}]}
    ats_srv = create_sdk_mcp_server(name="ats", version="1.0.0", tools=[ats_get])

    BGCHECK_KEY = os.environ.get("BGCHECK_API_KEY", "demo-key")              # credential FROM the environment

    @tool("get_background", "Get the background-check status for a candidate.", {"candidate_id": str})
    async def bg_get(args):
        return {"content": [{"type": "text",
                "text": f"cleared for {args['candidate_id']} (auth={BGCHECK_KEY[:4]}...)"}]}
    bg_srv = create_sdk_mcp_server(name="bg", version="1.0.0", tools=[bg_get])

    MULTI = ClaudeAgentOptions(model=MODEL, mcp_servers={"ats": ats_srv, "bg": bg_srv},
                               allowed_tools=["mcp__ats__get_candidate", "mcp__bg__get_background"])

    async def ask(prompt):
        async for m in query(prompt=prompt, options=MULTI):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, ToolUseBlock): print("  -> tool:", b.name.split("__")[-1], b.input)
                    elif isinstance(b, TextBlock) and b.text.strip(): print("  model:", b.text.strip()[:160])
    print("connected servers:", list(MULTI.mcp_servers), "-> all tools available at once")
except Exception:
    SDK_OK = False
    print("Agent SDK not available offline; the config and retry logic above were tested with Python.")

In [ ]:
# ===== run a request that spans both servers and hits a structured error =====
if RUN_LIVE and SDK_OK:
    print("--- spans both servers ---")
    run_async(lambda: ask("What stage is candidate C1 at, and is their background check cleared?"))
    print("--- malformed id returns a structured error the model corrects ---")
    run_async(lambda: ask("Get the stage for candidate 'C-1-7'."))
else:
    print("[offline] expected live: get_candidate (ATS) and get_background (bg) both fire from the")
    print("          two servers; a malformed id returns a validation error and the model retries 'C1'.")

---

### Anti-patterns to avoid

| anti-pattern | what to do instead |
|---|---|
| paste an API key straight into `.mcp.json` | reference it as `${VAR}` and keep the secret in your shell |
| commit `.env` or a filled config to Git | commit the `${VAR}` config; document required vars in the README |
| one giant server for everything | split by system (ATS, background) and connect several at once |
| return every failure as the same generic error | tag each with a category and `isRetryable` |
| throw an exception for a tool failure | return `isError: true` so the model can read and recover |
| retry a valid empty result | check `isError` and `resultCount`; an empty result is a success |
| retry a business or permission error | only retry `transient`; escalate or fix the others |

**Lesson:** `.mcp.json` (project) and `~/.claude.json` (user) declare your MCP servers, and
`${VAR}` keeps their secrets in the environment so the config is safe to share; connect several
servers and all their tools are available at once. And a good tool error is a recovery instruction:
signal failure with `isError`, keep it separate from a valid empty result, attach a **category**, an
**`isRetryable`** flag, and a recovery **description**, and one retry harness can retry transient
failures and route everything else to the right action.

---

## Recap - configure safely, fail well

| Piece | In this lab | Course topic |
|---|---|---|
| `.mcp.json` / `~/.claude.json` | project and user config | version-controlled and user-level config (Lab 1) |
| `${VAR}` | credentials from the environment | secure, portable credentials (Lab 1) |
| Multi-server | two servers in one options object | all MCP tools available at once (Lab 1) |
| `isError` + category | four structured error kinds | tool execution errors the model reads (Lab 2) |
| `isRetryable` + harness | retry transient, escalate the rest | retry logic from metadata (Lab 2) |
| Empty vs error | `resultCount: 0` is a success | do not retry a valid empty result (Lab 2) |

**Try it next:** add a third in-process server and expose its tool alongside the others. Then add a
`rate_limit` transient variant whose description includes a reset time, and have the harness respect
it.